# Day 55 — APIs & Containerization: Dockerize a prediction service
Objectives:
- Package your FastAPI model API (from Day 44).
- Create a minimal Dockerfile.
- Build and run locally; test with curl.
Note: You need Docker Desktop or a Docker runtime installed to build images.

## Project layout
````
ds-60day/
  app.py               # from Day 44 (FastAPI)
  model.joblib         # saved model
  requirements-api.txt # smaller requirements for API runtime
  Dockerfile
````

## Example requirements-api.txt
```text
fastapi
uvicorn
joblib
numpy
scikit-learn
pydantic
```


## Example Dockerfile
```Dockerfile
# syntax=docker/dockerfile:1
FROM python:3.12-slim
WORKDIR /app
COPY requirements-api.txt ./
RUN python -m pip install --no-cache-dir -r requirements-api.txt
COPY app.py model.joblib ./
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```
Build & run:
```bash
docker build -t ds-60day-api .
docker run --rm -p 8000:8000 ds-60day-api
```
Test:
```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"features": [5.1, 3.5, 1.4, 0.2]}'
```
PowerShell:
```powershell
$body = @{ features = @(5.1, 3.5, 1.4, 0.2) } | ConvertTo-Json
Invoke-RestMethod -Method Post -Uri 'http://127.0.0.1:8000/predict' -ContentType 'application/json' -Body $body
```


## Learner exercises and progressive hints

1. Create a slim dependency file containing only direct API runtime needs.
2. Add `GET /health` returning `{"status": "ok"}`.
3. Optionally push the image to a registry if you intentionally use a connected
   account.

### Progressive hints

1. Trace imports from `app.py` and the serialized pipeline. Rebuild in a clean
   image and run both endpoints.
2. Keep liveness cheap. A Docker `HEALTHCHECK` can use Python's standard
   `urllib.request` so a slim image does not need `curl`.
3. Use a non-secret image name/tag, authenticate through the registry's
   supported credential flow, scan the image, and never embed credentials in a
   layer. Registry upload is optional and networked.

### Additional mastery practice

Containerize a minimal, testable service without embedding secrets or privileged assumptions. Build, readiness, and runtime health have distinct contracts.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Layer and secret audit:** Create a `.dockerignore`, inspect image history, and prove that `.env`, Git metadata, notebooks, caches, and local artifacts are absent.
   **Progressive hint:** The build context is the first boundary. Deleting a secret in a later layer does not remove it from earlier layers.
5. **Least-privilege runtime:** Run the service as a non-root user with a read-only filesystem and an explicit writable temporary directory. Diagnose any write assumptions.
   **Progressive hint:** Create the user in the image, set ownership only where needed, and write transient files under an intentionally mounted/temp path.
6. **Health semantics:** Implement separate `/live` and `/ready` checks and a startup failure when the model manifest is incompatible. Test all three states.
   **Progressive hint:** Liveness answers whether the process can respond; readiness answers whether it can safely serve the declared model contract.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.


In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Layer and secret audit


# Practice 5 — Least-privilege runtime


# Practice 6 — Health semantics
